In [1]:
import pandas as pd
from tqdm import tqdm
import math
import json
import os
from datetime import datetime

import torch
from torch_geometric.loader import DataLoader

from extractor import PDBBindOrchestrator
from parsers.cnn_parser import CNNParser
from parsers.gnn_parser import GNNParser
from tokenizer import UniversalPDBBindDataset
from model import UniversalHybridModel
from encoders.cnn_encoder import FlexCNNBlock
from encoders.gnn_encoder import FlexGNNBlock
from encoders.original_quantum_encoder import QuantumReUploadingLayer
from evaluator import EValuator

In [2]:
def get_loss_bar(loss_val, bar_len=10):
    # Используем log10 для нормализации динамического диапазона
    # Добавляем 1e-9, чтобы избежать log(0)
    log_loss = math.log10(loss_val + 1e-9)
    
    # Масштабируем: допустим, мы ожидаем лосс от 50 (log ~1.7) до 0.1 (log -1)
    # Сделаем простую линейную закраску для диапазона log_loss от -1 до 2
    level = (log_loss + 1) / 3  # нормализуем в [0, 1]
    level = max(0, min(1, level)) # ограничиваем
    
    filled = int(level * bar_len)
    return "[" + "█" * filled + " " * (bar_len - filled) + "]"

In [3]:
prot_parser = CNNParser(is_ligand=False)
ligand_parser = CNNParser(is_ligand=True)
pocket_parser = GNNParser(is_ligand=False)
processor = PDBBindOrchestrator(prot_parser=prot_parser, lig_parser=ligand_parser, pock_parser=pocket_parser)
processor.extract_subset("refined")

df_refined = processor.build_dataset(subset="refined", fmt="pickle")
refined_dataset_full_path, refined_metadata_full_path = processor.full_path, processor.full_meta_path
df_core = processor.build_dataset(subset="core", fmt="pickle")
core_dataset_full_path, core_metadata_full_path = processor.full_path, processor.full_meta_path
del df_refined, df_core

Распаковка 4057 комплексов...


0it [00:00, ?it/s]

66444it [01:24, 785.09it/s]


Распаковка завершена.
Запуск параллельного парсинга на 8 ядрах...


100%|██████████| 4057/4057 [00:57<00:00, 70.82it/s] 


Успешно: 4056, Ошибок: 1
Counter({"ligand_parse_error: Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8": 1})
Метаданные сохранены в datasets/pdbbind_refined_protC_ligC_pockG_meta.json
Датасет сохранен в datasets/pdbbind_refined_protC_ligC_pockG.pickle (сжатие: None)
Запуск параллельного парсинга на 8 ядрах...


100%|██████████| 290/290 [00:04<00:00, 67.24it/s]


Успешно: 289, Ошибок: 1
Counter({"ligand_parse_error: Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8": 1})
Метаданные сохранены в datasets/pdbbind_core_protC_ligC_pockG_meta.json
Датасет сохранен в datasets/pdbbind_core_protC_ligC_pockG.pickle (сжатие: None)


In [4]:
print(f"Train dataset path: {refined_dataset_full_path}")
print(f"Train metadata path: {refined_metadata_full_path}")
print(f"Test dataset path: {core_dataset_full_path}")
print(f"Test metadata path: {core_metadata_full_path}")

name_train = os.path.basename(refined_dataset_full_path).split(".")[0]
name_test = os.path.basename(core_dataset_full_path).split(".")[0]
print(f"Train dataset name: {name_train}")
print(f"Test dataset name: {name_test}")

full_train_path = os.path.join(os.path.dirname(refined_dataset_full_path), f"{name_train}_train.pickle")
full_test_path = os.path.join(os.path.dirname(core_dataset_full_path), f"{name_test}_test.pickle")

df_refined = pd.read_pickle(refined_dataset_full_path)
df_core = pd.read_pickle(core_dataset_full_path)
train_df = df_refined[~df_refined['pdb_id'].isin(df_core['pdb_id'])]
train_df.to_pickle(full_train_path)

test_df = df_core
test_df.to_pickle(full_test_path)

Train dataset path: datasets/pdbbind_refined_protC_ligC_pockG.pickle
Train metadata path: datasets/pdbbind_refined_protC_ligC_pockG_meta.json
Test dataset path: datasets/pdbbind_core_protC_ligC_pockG.pickle
Test metadata path: datasets/pdbbind_core_protC_ligC_pockG_meta.json
Train dataset name: pdbbind_refined_protC_ligC_pockG
Test dataset name: pdbbind_core_protC_ligC_pockG


In [5]:
# 1. ЕДИНЫЙ ГЛОБАЛЬНЫЙ КОНФИГ
config = {
    "experiment_name": "CCG_GAT_AdamW",
    "dataset": {
        "train_path": full_train_path,
        "test_path": full_test_path,
        "refined_meta_path": refined_metadata_full_path,
        "core_meta_path": core_metadata_full_path,
        "max_prot": 1000,
        "max_lig": 150,
        "max_pock": 63,
        "batch_size": 32
    },
    "model": {
        "embed_dim": 128,
        "gnn_mode": "gat",
        "gat_heads": 4,
        "cnn_mode": "parallel",
        "n_qubits": 9,
        "q_layers": 8
    },
    "training": {
        "epochs": 20,
        "classic_lr": 0.001,
        "quantum_lr": 0.0001,
        "classic_optimizer": "AdamW",
        "quantum_optimizer": "SGD",
        "loss_fn": "MSELoss"
    }
}

In [ ]:
# 1. Инициализируем датасеты, чтобы узнать реальные размеры словарей, длины согласно статье.
train_ds = UniversalPDBBindDataset(full_train_path, max_prot=config['dataset']['max_prot'], max_lig=config['dataset']['max_lig'], max_pock=config['dataset']['max_pock'])
test_ds = UniversalPDBBindDataset(full_test_path, max_prot=config['dataset']['max_prot'], max_lig=config['dataset']['max_lig'], max_pock=config['dataset']['max_pock'])

config["dataset"]["prot_vocab"] = max(train_ds.prot_vocab.values()) + 1
config["dataset"]["lig_vocab"] = max(train_ds.lig_vocab.values()) + 1

# 3. Инициализация модели и железа
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HQDeepDTAF = UniversalHybridModel(
    protein_encoder = FlexCNNBlock(config['dataset']['prot_vocab'], config['model']['embed_dim'], mode=config['model']['cnn_mode']),
    ligand_encoder = FlexCNNBlock(config['dataset']['lig_vocab'], config['model']['embed_dim'], mode=config['model']['cnn_mode']),
    pocket_encoder = FlexCNNBlock(config['dataset']['prot_vocab'], config['model']['embed_dim'], mode=config['model']['cnn_mode']),
    quantum_encoder = QuantumReUploadingLayer(config['model']['n_qubits'], config['model']['q_layers'])
)

CustomNetwork = UniversalHybridModel(
    protein_encoder = FlexCNNBlock(config['dataset']['prot_vocab'], config['model']['embed_dim'], mode=config['model']['cnn_mode']),
    ligand_encoder = FlexCNNBlock(config['dataset']['lig_vocab'], config['model']['embed_dim'], mode=config['model']['cnn_mode']),
    pocket_encoder = FlexGNNBlock(in_channels=3, hidden_channels=config['model']['embed_dim'], out_channels=config['model']['embed_dim'], num_layers=3, conv_type=config['model']['gnn_mode'], heads=config['model']['gat_heads']),
    quantum_encoder = QuantumReUploadingLayer(config['model']['n_qubits'], config['model']['q_layers'])
)

model = CustomNetwork.to(device)
evaluator = EValuator(model, device)

# 4. Настройка обучения
# optimizer = optim.Adam(model.parameters(), lr=0.001)
classic_params = (
    list(model.protein_encoder.parameters()) + 
    list(model.ligand_encoder.parameters()) + 
    list(model.pocket_encoder.parameters()) +
    list(model.pre_quantum.parameters()) +
    list(model.regressor.parameters())
)
quantum_params = model.quantum_encoder.parameters()

# CNN и обвязка любят Adam за скорость
classic_opt_class = getattr(torch.optim, config['training']['classic_optimizer'])
classic_optimizer = classic_opt_class(classic_params, lr=config['training']['classic_lr'])
    
# Квантовое ядро часто лучше учится на SGD или Adagrad, 
# так как они меньше "шумят" в фазовом пространстве
# quantum_optimizer = optim.AdamW(quantum_params, lr=0.0001)
quantum_opt_class = getattr(torch.optim, config['training']['quantum_optimizer'])
quantum_optimizer = quantum_opt_class(quantum_params, lr=config['training']['quantum_lr'], momentum=0.9)

criterion = getattr(torch.nn, config['training']['loss_fn'])()

train_loader = DataLoader(train_ds, batch_size=config['dataset']['batch_size'], shuffle=True)
test_loader = DataLoader(test_ds, batch_size=config['dataset']['batch_size'], shuffle=False)

def train(epochs=config['training']['epochs'], show_plots=False, save_plots=True): # По статье авторы используют 20 эпох 
    print(f"Starting training on {device}...")
    # Создаем папку для эксперимента
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    exp_dir = f"runs/{config['experiment_name']}_{timestamp}"
    with open(refined_metadata_full_path, 'r', encoding='utf-8') as f:
        train_meta = json.load(f)
    with open(core_metadata_full_path, 'r', encoding='utf-8') as f:
        test_meta = json.load(f)
    config['dataset']['train_metadata'] = train_meta
    config['dataset']['test_metadata'] = test_meta
    os.makedirs(exp_dir, exist_ok=True)
    with open(f"{exp_dir}/config.json", 'w') as f:
        json.dump(config, f, indent=4)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        # Создаем обертку над лоадером
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch", leave=True)
        
        for i, (prot, lig, pock, y) in enumerate(pbar):
            prot, lig, pock, y = prot.to(device), lig.to(device), pock.to(device), y.to(device)
            
            classic_optimizer.zero_grad()
            quantum_optimizer.zero_grad()
            output = model(prot, lig, pock).view(-1)
            loss = criterion(output, y)
            loss.backward()
            classic_optimizer.step()
            quantum_optimizer.step()
            
            current_loss = loss.item()
            total_loss += current_loss
            
            # Генерируем визуальную полоску лосса
            l_bar = get_loss_bar(current_loss)
            
            # Выводим в tqdm: текущий лосс, средний и нашу полоску
            avg_loss = total_loss / (i + 1)
            pbar.set_postfix_str(f"Loss: {current_loss:.4f} {l_bar} Avg: {avg_loss:.4f}")

        # Валидация метрик (RMSE, Pearson R, CI) в конце эпохи
        rmse, r_val, ci_val, preds, targets = evaluator.evaluate(test_loader)
        model.history['train_loss'].append(avg_loss)
        model.history['val_rmse'].append(rmse)
        model.history['val_pearson'].append(r_val)
        model.history['val_ci'].append(ci_val)
        if r_val >= max(model.history['val_pearson']):
            model.history['best_y_true'] = targets.tolist()
            model.history['best_y_pred'] = preds.tolist()

        with open(f"{exp_dir}/history.json", 'w') as f:
            json.dump(model.history, f, indent=4)
        torch.save(model.state_dict(), f"{exp_dir}/model_epoch_{epoch}.pt")

        # Печатаем итоги эпохи (в статье используются именно эти метрики [cite: 515, 516, 521])
        print(f"   ∟ Valid: RMSE {rmse:.4f} | R {r_val:.4f} | CI {ci_val:.4f}")
        print("-" * 60)

    evaluator.plot_history(exp_dir, show=show_plots, save=save_plots)

In [7]:
train(10, show_plots=True, save_plots=True)

Starting training on cpu...


Epoch 1/10: 100%|██████████| 118/118 [03:14<00:00,  1.65s/batch, Loss: 19.4405 [███████   ] Avg: 35.0159]


TypeError: Object of type ndarray is not JSON serializable